# Other Utilities — Advanced Reference

| Utility | Purpose |
|---|---|
| `.cols(ascending=True/False/None)` | List column names sorted or in original order |
| `.handle_missing(fillna='.')` | Fill string NaN with a marker, numeric NaN with 0, strip whitespace |
| `.group_x(group, aggfunc, value, dropna)` | Add a group-level scalar back to every row (via `transform`) |
| `.clip()` | Copy DataFrame to clipboard (tab-separated, no index) |

---

In [1]:
import sys, os
_src = os.path.abspath(os.path.join(os.getcwd(), '..', 'src'))
if _src not in sys.path: sys.path.insert(0, _src)

import pytae as pt

penguins = pt.sample_data['penguins']
titanic = pt.sample_data['titanic']

## `cols()` — use to drive `select` dynamically
`cols()` returns a plain list — combine it with list comprehensions inside `select` for programmatic column selection.

In [2]:
# Alphabetical column list — useful for quick discovery
penguins.cols()

['bill_depth_mm',
 'bill_length_mm',
 'body_mass_g',
 'flipper_length_mm',
 'island',
 'sex',
 'species']

In [3]:
# Descending alphabetical sort — useful when column names form a natural reverse order
penguins.cols(ascending=False)

['species',
 'sex',
 'island',
 'flipper_length_mm',
 'body_mass_g',
 'bill_length_mm',
 'bill_depth_mm']

## `handle_missing()` — clean before aggregating
Fills string/category NaN with a visible marker (default `.`) and numeric NaN with `0`. Also strips leading/trailing whitespace from string columns.

In [4]:
# Default fillna='.': NaN sex becomes '.', NaN numerics become 0
# Then aggregate to see the '.' group alongside real groups
(penguins
 .handle_missing()
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,.
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,.
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [5]:
# Custom fillna marker — makes missing group explicit in reports
( penguins
 .handle_missing(fillna='Unknown')
)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,Unknown
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,Unknown
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


In [6]:
# Use a single-char marker like '$' to make missing data stand out in reports
( penguins
 .handle_missing(fillna='$')
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female
3,Adelie,Torgersen,0.0,0.0,0.0,0.0,$
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female
...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,0.0,0.0,0.0,0.0,$
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female


## `group_x()` — broadcast a group aggregate back to every row
Unlike `agg_df` which collapses rows, `group_x` **adds a column** to the original DataFrame using `transform`. Useful for computing a group value alongside row-level detail (e.g., share of group total, deviation from group mean).

In [7]:
# Default: count per group (aggfunc='n') — adds column 'n' to every row
( penguins
 .handle_missing()
 .group_x(group=['species','island' ])
 .select(['species', 'island', 'sex', 'body_mass_g', 'n'])
 .sample(10)
)


,species,island,sex,body_mass_g,n
62,Adelie,Biscoe,Female,3600.0,44
154,Chinstrap,Dream,Male,3650.0,68
283,Gentoo,Biscoe,Male,5650.0,124
117,Adelie,Torgersen,Male,3775.0,52
334,Gentoo,Biscoe,Female,4375.0,124
319,Gentoo,Biscoe,Male,5250.0,124
295,Gentoo,Biscoe,Male,5800.0,124
58,Adelie,Biscoe,Female,2850.0,44
122,Adelie,Torgersen,Female,3450.0,52
119,Adelie,Torgersen,Male,3325.0,52


In [8]:
# instead of count, group_x with value + aggfunc: add group mean of body_mass_g to every row
(penguins
 .group_x(group=['species'], value='body_mass_g', aggfunc='mean')
 .rename(columns={'x': 'species_mean_mass'})
 .sample(10)
)

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,species_mean_mass
314,Gentoo,Biscoe,44.5,14.7,214.0,4850.0,Female,5076.016260
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,None,5076.016260
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,3700.662252
307,Gentoo,Biscoe,51.3,14.2,218.0,5300.0,Male,5076.016260
337,Gentoo,Biscoe,48.8,16.2,222.0,6000.0,Male,5076.016260
214,Chinstrap,Dream,45.7,17.0,195.0,3650.0,Female,3733.088235
248,Gentoo,Biscoe,48.2,14.3,210.0,4600.0,Female,5076.016260
190,Chinstrap,Dream,46.9,16.6,192.0,2700.0,Female,3733.088235
99,Adelie,Dream,43.2,18.5,192.0,4100.0,Male,3700.662252
158,Chinstrap,Dream,46.1,18.2,178.0,3250.0,Female,3733.088235


In [9]:
#with dropna=False
penguins.group_x(group=['species', 'sex'],dropna=False)


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,n
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,73
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,73
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,73
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,None,6
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,73
...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,None,5
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,58
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,61
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,58


In [10]:
#with default dropna=True
penguins.group_x(group=['species', 'sex'])

,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,n
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,73.0
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,73.0
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,73.0
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,None,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,73.0
...,...,...,...,...,...,...,...,...
339,Gentoo,Biscoe,NaN,NaN,NaN,NaN,None,NaN
340,Gentoo,Biscoe,46.8,14.3,215.0,4850.0,Female,58.0
341,Gentoo,Biscoe,50.4,15.7,222.0,5750.0,Male,61.0
342,Gentoo,Biscoe,45.2,14.8,212.0,5200.0,Female,58.0


## `clip()` — copy to clipboard
`clip()` copies the DataFrame to the system clipboard as tab-separated text (no index). Paste directly into Excel, Google Sheets, or any text editor. Invisible in notebook output — run it and then paste.

In [11]:
# clip() copies the current DataFrame to clipboard — paste into Excel/Sheets
# Run this cell then Cmd+V in any spreadsheet application
(penguins
 .select(['species', 'island', 'bill_length_mm', 'body_mass_g'])
 .agg_df(aggfunc=['mean', 'n'])
 .clip()
)